# 🐍 Day 10 · Python Basics — Loops, Regex & Tool Dispatch (Agent From Scratch)

> **Goal of this notebook:** An agent is a loop: the LLM writes `Action: ...`, Python runs that tool, and the result goes back to the LLM. Build that loop yourself.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. A dictionary of tools
2. Calling a function chosen by name
3. Regular expressions (regex) to read the LLM's text
4. `try / except` — handling errors
5. A `while` loop with a step limit
6. 🔗 A complete mini ReAct agent (with a pretend LLM)

## 1. A dictionary of tools
The keys are tool **names**, the values are the **functions** themselves.

In [ ]:
def get_weather(city):
    fake_data = {"delhi": "35°C sunny", "london": "12°C rainy"}
    return fake_data.get(city.lower(), "unknown city")

def calculator(expression):
    # only digits and + - * / ( ) . are allowed, to keep eval safe
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "invalid expression"
    return str(eval(expression))

tools = {
    "get_weather": get_weather,
    "calculator": calculator,
}
print(list(tools.keys()))

## 2. Calling a function chosen by name

In [ ]:
tool_name = "get_weather"
tool_input = "Delhi"

func = tools[tool_name]          # look up the function
print(func(tool_input))          # call it

print(tools["calculator"]("12 * (3 + 2)"))

## 3. Regular expressions (regex)
The LLM answers with text like:
```
Thought: I need the weather
Action: get_weather
Action Input: London
```
A **regex** is a pattern used to find text. `re.search` finds the first match; `( )` *captures* a part.

In [ ]:
import re

llm_text = "Thought: I need the weather\nAction: get_weather\nAction Input: London"

action = re.search(r"Action: (.*)", llm_text)          # (.*) = capture everything after "Action: "
action_input = re.search(r"Action Input: (.*)", llm_text)

print(action.group(1))          # get_weather
print(action_input.group(1))    # London

# Some common regex pieces:  \d = digit,  \w = letter/digit,  + = one or more,  .* = anything
print(re.findall(r"\d+", "Order 66 costs 250 rupees"))   # ['66', '250']

## 4. `try / except` — don't crash on errors
If the LLM asks for a tool that does not exist, we catch the error and tell it politely.

In [ ]:
def run_tool(name, value):
    try:
        return tools[name](value)
    except KeyError:
        return f"Error: there is no tool called '{name}'"

print(run_tool("calculator", "2 + 2"))
print(run_tool("send_email", "hi"))

## 5. A `while` loop with a step limit
Agents must have a **maximum number of steps**, otherwise a confused LLM could loop forever.

In [ ]:
step = 0
max_steps = 3
while step < max_steps:
    step += 1                  # same as step = step + 1
    print("Thinking... step", step)
print("Stopped after", step, "steps")

## 6. 🔗 A complete mini ReAct agent
We use a **pretend LLM** (a list of prepared answers) so you can see the full Thought → Action → Observation loop without an API key.

In [ ]:
pretend_llm_answers = [
    "Thought: I should check the weather.\nAction: get_weather\nAction Input: Delhi",
    "Thought: Now convert 35 to Fahrenheit.\nAction: calculator\nAction Input: 35 * 9 / 5 + 32",
    "Thought: I now know the final answer.\nFinal Answer: Delhi is 35°C, which is 95°F.",
]

def fake_llm(prompt, step):
    return pretend_llm_answers[step]     # a real agent calls Gemini / GPT here

question = "What is the weather in Delhi in Fahrenheit?"
scratchpad = f"Question: {question}\n"

for step in range(5):                          # max 5 steps
    answer = fake_llm(scratchpad, step)
    scratchpad += answer + "\n"
    if "Final Answer:" in answer:
        print("✅", answer.split("Final Answer:")[1].strip())
        break
    name = re.search(r"Action: (.*)", answer).group(1).strip()
    value = re.search(r"Action Input: (.*)", answer).group(1).strip()
    observation = run_tool(name, value)
    print(f"🔧 {name}({value!r}) -> {observation}")
    scratchpad += f"Observation: {observation}\n"

print("\n--- Full scratchpad sent to the LLM ---")
print(scratchpad)

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Add a new tool `word_count(text)` to the `tools` dictionary and call it through `run_tool`.
2. Use `re.findall` to get all words that start with a capital letter in `"Ravi met Sara in Paris"` (hint: `r"[A-Z]\w+"`).
3. Change `max_steps` in section 5 to 5 and run it again.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
def word_count(text):
    return len(text.split())
tools["word_count"] = word_count
print(run_tool("word_count", "agents are just loops"))

print(re.findall(r"[A-Z]\w+", "Ravi met Sara in Paris"))   # ['Ravi', 'Sara', 'Paris']
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Tools dictionary | `tools = {'calc': calculator}` |
| Call by name | `tools[name](value)` |
| Regex search | `re.search(r'Action: (.*)', t).group(1)` |
| Find all | `re.findall(r'\d+', t)` |
| Catch errors | `try: ... except KeyError:` |
| Step limit | `for step in range(5): ... break` |
| Add to text | `scratchpad += more` |

## ➡️ What's next?
You now know the Python used in **Day_10_Agents_From_Scratch_ReAct**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀